# Chest X-Ray Multi-Condition Detection — Exploratory Data Analysis

**Team Coffee and Code** · DataML Technical Championship

This notebook covers Week 1 exploration: understanding the label structure, the images
themselves, and looking for early evidence that simple hand-computed statistics carry
signal about the five target conditions.

**Constraint reminder:** classical ML only. No CNNs, no deep learning, no pretrained
models or pretrained feature extractors at any stage of the pipeline.

**Before running:** place the dataset so the layout matches the `Config` cell below.
The image folders are deliberately *not* committed to the repo (see `.gitignore`).

```
ML-Model-Varchas/
├── data/
│   ├── train/            # training images
│   ├── test/             # test images (unlabelled)
│   └── train_labels.csv
└── code/
    └── 01_eda.ipynb      # this notebook
```


## 1. Setup

In [ ]:
import os
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("numpy", np.__version__)
print("pandas", pd.__version__)

### Config

Everything path-related lives here. If your local layout differs, this is the only
cell you should need to touch.

In [ ]:
# Notebook lives in code/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()

DATA_DIR   = PROJECT_ROOT / "data"
TRAIN_DIR  = DATA_DIR / "train"
TEST_DIR   = DATA_DIR / "test"
LABELS_CSV = DATA_DIR / "train_labels.csv"

CONDITIONS = ["Atelectasis", "Effusion", "Infiltration", "Nodule", "Pneumothorax"]

for p, name in [(DATA_DIR, "data dir"), (TRAIN_DIR, "train dir"),
                (TEST_DIR, "test dir"), (LABELS_CSV, "labels csv")]:
    print(f"{'OK  ' if p.exists() else 'MISS'} {name:11s} -> {p}")

## 2. Labels: load and sanity-check

In [ ]:
labels = pd.read_csv(LABELS_CSV)

print("shape:", labels.shape)
print("columns:", list(labels.columns))
labels.head()

In [ ]:
# Identify the ID column: whichever column is not one of the five conditions.
id_candidates = [c for c in labels.columns if c not in CONDITIONS]
print("Non-condition columns:", id_candidates)

ID_COL = id_candidates[0]
print("Using ID column:", ID_COL)

# Basic integrity checks
print("\nduplicate IDs :", labels[ID_COL].duplicated().sum())
print("null values    :", labels.isna().sum().sum())
print("\nvalue counts per condition column (expect only 0/1):")
for c in CONDITIONS:
    print(f"  {c:14s} {dict(labels[c].value_counts())}")

## 3. Class distribution

Chest X-ray datasets are almost always heavily imbalanced. How imbalanced determines
a lot downstream: whether we need class weighting, what threshold we pick, and — most
importantly — which evaluation metric is honest. Accuracy is meaningless if a condition
appears in 4% of scans.

In [ ]:
prevalence = labels[CONDITIONS].mean().sort_values(ascending=False)
counts = labels[CONDITIONS].sum().loc[prevalence.index]

summary = pd.DataFrame({
    "positive_count": counts.astype(int),
    "negative_count": (len(labels) - counts).astype(int),
    "prevalence_%": (prevalence * 100).round(2),
    "imbalance_ratio": ((len(labels) - counts) / counts).round(1),
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(prevalence.index[::-1], (prevalence * 100)[::-1], color="#4C72B0")
ax.set_xlabel("Prevalence (% of training images)")
ax.set_title("Condition prevalence — training set")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax.set_xlim(0, max(prevalence * 100) * 1.2)
plt.tight_layout()
plt.show()

## 4. Multi-label structure

This is a multi-label problem, not multi-class — an image can carry several conditions
at once, or none. Two things worth knowing:

1. **How many labels does a typical image carry?** If most images are all-zero, the
   "no finding" case dominates and our features need to separate normal from abnormal
   before they separate condition from condition.
2. **Which conditions co-occur?** Strong co-occurrence means the five binary problems
   are not independent, and a classifier chain or stacked approach may beat five
   isolated one-vs-rest models.

In [ ]:
labels["n_conditions"] = labels[CONDITIONS].sum(axis=1)

dist = labels["n_conditions"].value_counts().sort_index()
dist_df = pd.DataFrame({
    "images": dist,
    "share_%": (dist / len(labels) * 100).round(2),
})
dist_df.index.name = "conditions_present"
print(dist_df)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(dist.index.astype(str), dist.values, color="#DD8452")
ax.set_xlabel("Number of conditions present in one image")
ax.set_ylabel("Image count")
ax.set_title("How many labels does an image carry?")
ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Co-occurrence: how often do two conditions appear on the same image?
co = labels[CONDITIONS].T.dot(labels[CONDITIONS])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(co.values, cmap="Blues")
ax.set_xticks(range(len(CONDITIONS)), CONDITIONS, rotation=45, ha="right")
ax.set_yticks(range(len(CONDITIONS)), CONDITIONS)
ax.set_title("Co-occurrence counts (diagonal = total positives)")
ax.grid(False)

for i in range(len(CONDITIONS)):
    for j in range(len(CONDITIONS)):
        v = co.values[i, j]
        ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=8,
                color="white" if v > co.values.max() * 0.5 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Lift: P(A and B) / (P(A) * P(B)). Lift > 1 means the pair co-occurs more than chance.
p = labels[CONDITIONS].mean()
joint = co / len(labels)
expected = np.outer(p, p)

lift_arr = np.array(joint.values / expected, dtype=float, copy=True)
np.fill_diagonal(lift_arr, np.nan)
lift = pd.DataFrame(lift_arr, index=CONDITIONS, columns=CONDITIONS)

print("Pairwise lift (>1 = co-occur more often than independence would predict)\n")
lift.round(2)

## 5. Image inventory

Before computing any feature we need to know what we are actually working with:
how many files, what dimensions, what bit depth, whether they are truly greyscale,
and whether every labelled ID has a matching file on disk.

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def list_images(folder):
    if not folder.exists():
        return []
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS)

train_files = list_images(TRAIN_DIR)
test_files  = list_images(TEST_DIR)

print(f"train images on disk : {len(train_files):,}")
print(f"test images on disk  : {len(test_files):,}")
print(f"rows in labels csv   : {len(labels):,}")

if train_files:
    print("\nexample filenames:", [f.name for f in train_files[:3]])
    print("example label IDs :", labels[ID_COL].head(3).tolist())

In [ ]:
# Do the labelled IDs line up with files on disk?
disk_names = {f.name for f in train_files}
disk_stems = {f.stem for f in train_files}

label_ids = labels[ID_COL].astype(str)

matched_full  = label_ids.isin(disk_names).sum()
matched_stem  = label_ids.isin(disk_stems).sum()

print(f"IDs matching a filename WITH extension    : {matched_full:,} / {len(labels):,}")
print(f"IDs matching a filename WITHOUT extension : {matched_stem:,} / {len(labels):,}")

# Pick whichever convention matches, and build an ID -> path lookup.
if matched_full >= matched_stem:
    path_lookup = {f.name: f for f in train_files}
else:
    path_lookup = {f.stem: f for f in train_files}

missing = [i for i in label_ids if i not in path_lookup]
print(f"\nlabelled IDs with no image file: {len(missing)}")
if missing[:5]:
    print("  e.g.", missing[:5])

unlabelled = [k for k in path_lookup if k not in set(label_ids)]
print(f"image files with no label row  : {len(unlabelled)}")

In [ ]:
# Inspect a sample of images for dimensions / mode / dtype.
SAMPLE_N = min(300, len(train_files))
sample_files = list(rng.choice(train_files, size=SAMPLE_N, replace=False)) if train_files else []

records = []
for f in sample_files:
    try:
        with Image.open(f) as im:
            records.append({
                "file": f.name,
                "width": im.size[0],
                "height": im.size[1],
                "mode": im.mode,
                "format": im.format,
                "kb": round(f.stat().st_size / 1024, 1),
            })
    except Exception as e:
        records.append({"file": f.name, "error": str(e)})

meta = pd.DataFrame(records)
print(f"Inspected {len(meta)} images\n")
print("modes     :", dict(Counter(meta["mode"].dropna())))
print("formats   :", dict(Counter(meta["format"].dropna())))
print("dimensions:", dict(Counter(zip(meta["width"], meta["height"])).most_common(5)))
print()
meta[["width", "height", "kb"]].describe().round(1)

> **Read the output above carefully.** If every image is the same size and mode, feature
> extraction is straightforward. If sizes vary, decide now on a resize convention — and
> note that resizing changes texture statistics, so whatever you pick has to be applied
> identically to train and test.

## 6. Look at the actual images

Numbers only get you so far. Before designing features it is worth genuinely *looking*
at scans for each condition and asking: what visually distinguishes this from a normal
scan, and where in the image does that difference live?

In [ ]:
def load_gray(path, size=None):
    """Load an image as a float array in [0, 1], greyscale."""
    with Image.open(path) as im:
        im = im.convert("L")
        if size is not None:
            im = im.resize(size, Image.BILINEAR)
        return np.asarray(im, dtype=np.float32) / 255.0


def show_examples(condition, n=5, exclusive=True):
    """Show n training images positive for `condition`.
    exclusive=True picks images where it is the ONLY condition present."""
    mask = labels[condition] == 1
    if exclusive:
        mask &= labels["n_conditions"] == 1
    subset = labels[mask]

    if len(subset) == 0:
        print(f"No {'exclusive ' if exclusive else ''}examples for {condition}")
        return

    picks = subset.sample(min(n, len(subset)), random_state=RANDOM_SEED)
    fig, axes = plt.subplots(1, len(picks), figsize=(3 * len(picks), 3.4))
    axes = np.atleast_1d(axes)

    for ax, (_, row) in zip(axes, picks.iterrows()):
        path = path_lookup.get(str(row[ID_COL]))
        if path is None:
            ax.axis("off"); continue
        ax.imshow(load_gray(path), cmap="gray")
        ax.set_title(str(row[ID_COL]), fontsize=7)
        ax.axis("off")

    tag = "only condition" if exclusive else "present"
    fig.suptitle(f"{condition} ({tag})", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


# Healthy baseline for comparison
def show_negatives(n=5):
    subset = labels[labels["n_conditions"] == 0]
    if len(subset) == 0:
        print("No all-negative images"); return
    picks = subset.sample(min(n, len(subset)), random_state=RANDOM_SEED)
    fig, axes = plt.subplots(1, len(picks), figsize=(3 * len(picks), 3.4))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, picks.iterrows()):
        path = path_lookup.get(str(row[ID_COL]))
        if path is None:
            ax.axis("off"); continue
        ax.imshow(load_gray(path), cmap="gray")
        ax.set_title(str(row[ID_COL]), fontsize=7)
        ax.axis("off")
    fig.suptitle("No findings (all five negative)", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
show_negatives(5)

for cond in CONDITIONS:
    show_examples(cond, n=5, exclusive=True)

**Questions to answer while looking at these — write your answers down, they become
the justification section of the write-up:**

- *Effusion* — fluid pools at the lung base. Does the lower zone look denser/whiter?
  Is the costophrenic angle blunted?
- *Pneumothorax* — air in the pleural space. Is there a region that is unusually *dark*
  and texture-free, with a visible lung edge?
- *Nodule* — small round opacity. Is it a local blob rather than a global change?
  This argues for local/patch features, not whole-image summaries.
- *Infiltration* — patchy hazy opacity, often diffuse. A texture-variance story.
- *Atelectasis* — collapsed lung tissue, volume loss, shifted structures. Possibly
  an asymmetry story between left and right.

Note how many of these are **regional**, not global. That is the single biggest hint
for the feature engineering stage.

## 7. Pixel intensity: does anything separate at all?

The crudest possible probe. If whole-image intensity statistics already shift between
positive and negative cases for a condition, that condition has *some* global signal.
If they do not, that condition will need regional or texture features — which is useful
to know before spending days on it.

This is not a model. It is a check on whether the cheapest features are worth including.

In [ ]:
# Compute simple global stats on a subsample. Keep it small — this is a probe.
PROBE_N = min(1500, len(labels))
RESIZE_TO = (256, 256)

probe_ids = labels.sample(PROBE_N, random_state=RANDOM_SEED)

rows = []
for _, r in probe_ids.iterrows():
    path = path_lookup.get(str(r[ID_COL]))
    if path is None:
        continue
    try:
        img = load_gray(path, size=RESIZE_TO)
    except Exception:
        continue

    flat = img.ravel()
    rows.append({
        ID_COL: r[ID_COL],
        "mean": flat.mean(),
        "std": flat.std(),
        "p05": np.percentile(flat, 5),
        "p95": np.percentile(flat, 95),
        "skew": float(((flat - flat.mean()) ** 3).mean() / (flat.std() ** 3 + 1e-8)),
        "dark_frac": float((flat < 0.20).mean()),
        "bright_frac": float((flat > 0.80).mean()),
        **{c: r[c] for c in CONDITIONS},
    })

probe = pd.DataFrame(rows)
print(f"computed stats for {len(probe):,} images")
probe.head()

In [ ]:
STATS = ["mean", "std", "p05", "p95", "skew", "dark_frac", "bright_frac"]

# For each condition, compare stat distributions between positive and negative groups.
# Cohen's d gives a scale-free sense of separation.
def cohens_d(a, b):
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / (pooled + 1e-8)

effect = pd.DataFrame(index=STATS, columns=CONDITIONS, dtype=float)
for cond in CONDITIONS:
    pos = probe[probe[cond] == 1]
    neg = probe[probe[cond] == 0]
    for s in STATS:
        effect.loc[s, cond] = cohens_d(pos[s], neg[s])

print("Cohen's d — positive vs negative, per global statistic")
print("|d| < 0.2 negligible · 0.2-0.5 small · 0.5-0.8 medium · >0.8 large\n")
effect.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
vmax = np.nanmax(np.abs(effect.values)) or 1.0
im = ax.imshow(effect.values.astype(float), cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(CONDITIONS)), CONDITIONS, rotation=45, ha="right")
ax.set_yticks(range(len(STATS)), STATS)
ax.set_title("Effect size of global intensity stats (Cohen's d)")
ax.grid(False)
for i in range(len(STATS)):
    for j in range(len(CONDITIONS)):
        v = effect.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

In [ ]:
# Look at the distributions directly for the strongest pairing found above.
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(4 * len(CONDITIONS), 3.2), sharey=True)
STAT_TO_PLOT = "mean"   # change this to whichever stat showed the largest |d|

for ax, cond in zip(np.atleast_1d(axes), CONDITIONS):
    pos = probe.loc[probe[cond] == 1, STAT_TO_PLOT]
    neg = probe.loc[probe[cond] == 0, STAT_TO_PLOT]
    bins = np.linspace(probe[STAT_TO_PLOT].min(), probe[STAT_TO_PLOT].max(), 40)
    ax.hist(neg, bins=bins, alpha=0.55, density=True, label="negative", color="#4C72B0")
    ax.hist(pos, bins=bins, alpha=0.55, density=True, label="positive", color="#C44E52")
    ax.set_title(cond, fontsize=10)
    ax.set_xlabel(STAT_TO_PLOT)

axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
plt.suptitle(f"Distribution of '{STAT_TO_PLOT}' by label", y=1.03)
plt.tight_layout()
plt.show()

## 8. Regional probe — does location matter?

Section 6 suggested most of these conditions are *regional*. Test that cheaply: split
each image into a coarse grid and check whether per-zone intensity separates better
than the whole-image number did.

If a zone-level statistic beats the global one, that is direct evidence for building
regional features — and it is exactly the kind of reasoning the judges said they are
looking for.

In [ ]:
GRID = (3, 3)   # rows, cols

def zone_means(img, grid=GRID):
    rows, cols = grid
    h, w = img.shape
    out = []
    for i in range(rows):
        for j in range(cols):
            block = img[i * h // rows:(i + 1) * h // rows,
                        j * w // cols:(j + 1) * w // cols]
            out.append(block.mean())
    return out

zone_names = [f"z{i}{j}" for i in range(GRID[0]) for j in range(GRID[1])]

zrows = []
for _, r in probe_ids.iterrows():
    path = path_lookup.get(str(r[ID_COL]))
    if path is None:
        continue
    try:
        img = load_gray(path, size=RESIZE_TO)
    except Exception:
        continue
    zrows.append({ID_COL: r[ID_COL], **dict(zip(zone_names, zone_means(img))),
                  **{c: r[c] for c in CONDITIONS}})

zdf = pd.DataFrame(zrows)
print(f"zone stats for {len(zdf):,} images, {len(zone_names)} zones")

zeff = pd.DataFrame(index=zone_names, columns=CONDITIONS, dtype=float)
for cond in CONDITIONS:
    pos, neg = zdf[zdf[cond] == 1], zdf[zdf[cond] == 0]
    for z in zone_names:
        zeff.loc[z, cond] = cohens_d(pos[z], neg[z])

zeff.round(3)

In [ ]:
# Show each condition's zone effect map as a 3x3 image — where does the signal live?
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(3.1 * len(CONDITIONS), 3.4))
vmax = np.nanmax(np.abs(zeff.values)) or 1.0

for ax, cond in zip(np.atleast_1d(axes), CONDITIONS):
    grid_vals = zeff[cond].values.astype(float).reshape(GRID)
    im = ax.imshow(grid_vals, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(cond, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for i in range(GRID[0]):
        for j in range(GRID[1]):
            ax.text(j, i, f"{grid_vals[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes, shrink=0.7, label="Cohen's d")
plt.suptitle("Where in the image does each condition show up?", y=1.04)
plt.show()

print("\nBest global |d| per condition :")
print(effect.abs().max().round(3).to_string())
print("\nBest zone   |d| per condition :")
print(zeff.abs().max().round(3).to_string())

## 9. Test set check

The test images get the identical pipeline, so confirm now that they look the same as
training images. A dimension or mode mismatch discovered in Week 4 is a bad day.

In [ ]:
test_records = []
for f in (list(rng.choice(test_files, size=min(200, len(test_files)), replace=False))
          if test_files else []):
    try:
        with Image.open(f) as im:
            test_records.append({"width": im.size[0], "height": im.size[1], "mode": im.mode})
    except Exception:
        pass

tmeta = pd.DataFrame(test_records)
if len(tmeta):
    print("TEST  modes     :", dict(Counter(tmeta["mode"])))
    print("TEST  dimensions:", dict(Counter(zip(tmeta["width"], tmeta["height"])).most_common(5)))
    print()
    print("TRAIN modes     :", dict(Counter(meta["mode"].dropna())))
    print("TRAIN dimensions:", dict(Counter(zip(meta["width"], meta["height"])).most_common(5)))
else:
    print("No test images found — check TEST_DIR in the Config cell.")

## 10. Findings and next steps

*Fill this in as you go — these notes feed sections 4 and 5 of the README write-up.*

### What we found

- **Prevalence / imbalance:** …
- **Multi-label structure:** … (average labels per image, strongest co-occurring pair)
- **Image properties:** … (dimensions, mode, any inconsistencies)
- **Global intensity signal:** … (which conditions separate, which do not)
- **Regional signal:** … (did zone features beat global ones? where?)

### What this implies for feature engineering

Directions worth testing next, given classical-ML-only:

1. **Regional intensity statistics** — extend the 3×3 probe to a finer grid, and to
   anatomically motivated zones (upper/mid/lower, left/right) rather than a naive grid.
2. **Texture descriptors** — GLCM (contrast, homogeneity, energy, correlation) and
   Local Binary Patterns. Infiltration and atelectasis are texture changes more than
   brightness changes, so these should matter where plain intensity failed.
3. **Edge and gradient features** — HOG, or edge-density per zone. Nodules are small
   round opacities; edge structure should catch what a mean does not.
4. **Left–right asymmetry** — flip the image and difference it. Atelectasis and
   pneumothorax are usually unilateral, so asymmetry is a principled feature here.
5. **Histogram shape** — multi-bin intensity histograms per zone, rather than a
   handful of summary moments.

### Open questions

- Resize convention: what size, and does it destroy texture signal? Test at two sizes.
- Do we build one model per condition, or exploit the co-occurrence structure?
- Which metric do we report? Per-condition AUC plus macro-average is the honest choice
  given the imbalance — accuracy would be misleading.

### Before moving on

- [ ] Decide and freeze the train/validation split (stratify on the rarest condition)
- [ ] Save the split indices so every experiment is comparable
- [ ] Build `02_features.ipynb` — feature extraction, cached to disk
- [ ] Build `03_baseline.ipynb` — logistic regression on these probe features, to set
      the number every later model has to beat
